In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from wassa_functions import allen_reconstruction_comparison
from wassa_plots import make_violin
from wassa_metrics import WassDist
from wassa_training import get_training_parameters, learn_motifs
import matplotlib.pyplot as plt
import scipy, torch, glob
import numpy as np
from wassa import WassA
from tqdm.notebook import tqdm

In [ ]:
def paired_permutation_test(err_id, err_ood, n_resamples=10000, random_state=0, alternative="two-sided"):
    """
    Test de permutation apparié ID vs OOD pour une souris.
    """
    err_id = np.asarray(err_id)
    err_ood = np.asarray(err_ood)
    assert err_id.shape == err_ood.shape, "ID et OOD doivent être appariés"

    # Statistique : moyenne des différences
    def statistic(x, y):
        return np.mean(x-y)


    res = scipy.stats.permutation_test(
        data=(err_id, err_ood),
        statistic=statistic,
        permutation_type="samples",  # IMPORTANT pour données appariées
        alternative=alternative,
        n_resamples=n_resamples,
        random_state=random_state
    )

    return res.statistic, res.pvalue

## Get statistics from real data

In [ ]:
dataset_name = 'allen_data_by_image'
device = 'cpu'
dataset_path = '../../'+dataset_name+'/*'

In [ ]:
training_parameters_emd = {
    'kernel_size' : (1, 1, 1),
    'loss_type' : 'emd',
    'activation' : 'emd like',
    'sigmoid' : False,
    'kernels_norm' : 2,
    'N_learnsteps' : 2000,
    'learning_rate' : .01,
    'penalty_type' : [None],
    'lambda' : [0],
    'batch_size' : None,
    'do_bias' : False,
    'weight_init' : 'flat',
    'normalize_input' : True,
}

In [ ]:
saving_path = '../../wassa_copy/2025-09-30_static/results/wassa_allen_'
metric_names = ['training loss', 'testing loss', 'activations', 'explained variance']

In [ ]:
results_emd = allen_reconstruction_comparison(dataset_name,training_parameters_emd,metric_names,saving_path=saving_path,device=device)

In [ ]:
training_parameters_mse = {
    'kernel_size' : (1, 1, 1),
    'loss_type' : 'mse',
    'activation' : 'conv',
    'sigmoid' : False,
    'kernels_norm' : 2,
    'N_learnsteps' : 2000,
    'learning_rate' : .05,
    'penalty_type' : [None],
    'lambda' : [0],
    'batch_size' : None,
    'do_bias' : False,
    'weight_init' : 'flat',
    'normalize_input' : True
}

In [ ]:
results_mse = allen_reconstruction_comparison(dataset_name,training_parameters_mse,metric_names,saving_path=saving_path,device=device)

In [ ]:
training_parameters_frs = {
    'kernel_size' : (1, 1, 1),
    'loss_type' : 'mse',
    'activation' : 'conv',
    'sigmoid' : False,
    'kernels_norm' : 2,
    'N_learnsteps' : 2000,
    'learning_rate' : .05,
    'penalty_type' : [None],
    'lambda' : [0],
    'batch_size' : None,
    'do_bias' : False,
    'weight_init' : 'flat',
    'normalize_input' : False
}

In [ ]:
results_frs = allen_reconstruction_comparison(dataset_name,training_parameters_mse,metric_names,frs=True,saving_path=saving_path,device=device)

In [ ]:
all_metrics = ['MSE(id)-MSE(ood)', 'EMD(id)-EMD(ood)', 'EV(id)-EV(ood)']
all_results = [results_mse, results_emd, results_frs]
all_colors = ['darkolivegreen', 'blue', 'orange']
fig, ax = plt.subplots(len(all_metrics),3,figsize=(20,9))

separation_coef = .5
offset = 1
significants = [5e-2,1e-2,1e-3,1e-4]

p_values = np.zeros([32,3,3])
for i in range(len(all_metrics)):
    for ind_results, results in enumerate(all_results):
        reshaped = results[:,i].reshape(32,100,3).swapaxes(0,2)
        ax[i,ind_results] = make_violin(ax[i,ind_results],reshaped[1]-reshaped[2],all_colors[ind_results], separation_coef=separation_coef, offset=offset)
        for a in range(p_values.shape[0]):
            if i == 2:
                t_stat, p_value = paired_permutation_test(reshaped[1,:,a],reshaped[2,:,a],alternative='greater')
            else:
                t_stat, p_value = paired_permutation_test(reshaped[1,:,a],reshaped[2,:,a],alternative='less')
            p_values[a,i,ind_results] = p_value
        ax[i,ind_results].set_xticks([])
        ax[i,ind_results].ticklabel_format(axis='both', style='sci', scilimits=(-2,1))
        ax[i,ind_results].hlines(0,0,p_values.shape[0]*separation_coef+1,linestyles='dashed',colors='grey')
        ax[i,ind_results].set_xlim([separation_coef,p_values.shape[0]*separation_coef+1])
        ax[2,ind_results].set_xlabel('animal id',fontsize=20)
    ax[i,0].set_ylabel(all_metrics[i],fontsize=20)

pvals_fdr_frs = scipy.stats.false_discovery_control(p_values[:,[0,2],2].flatten(), method="bh").reshape(p_values.shape[0],2)
pvals_fdr_seq = scipy.stats.false_discovery_control(p_values[:,:,[0,1]].flatten(), method="bh").reshape(p_values.shape[0],3,2)

amplitudes = np.array([[3.5e-4, 3.5e-4], [3.5e-2,3.5e-2], [4e-4, 3.5e-5]])

for i_loss in range(pvals_fdr_seq.shape[1]):
    for i_train in range(pvals_fdr_seq.shape[2]):
        amp = amplitudes[i_loss,i_train]
        for a in range(p_values.shape[0]):
            n = 4
            ind = -1
            signific = significants[ind]
            while pvals_fdr_seq[a,i_loss,i_train]>significants[ind] and ind+n>0:
                ind-=1
            if pvals_fdr_seq[a,i_loss,i_train]<significants[0]:
                ax[i_loss,i_train].scatter([offset+a*separation_coef for i in range(n+ind+1)],[amp-i*amp*.1 for i in range(n+ind+1)],marker='$*$',color='black', alpha = .3)

for i_loss in range(pvals_fdr_frs.shape[1]):
    amp = 3.5e-1 if i_loss==1 else .085
    for a in range(p_values.shape[0]):
        n = 4
        ind = -1
        while pvals_fdr_frs[a,i_loss]>significants[ind] and ind+n>0:
            ind-=1
        if pvals_fdr_frs[a,i_loss]<significants[0]:
            ax[i_loss*2,2].scatter([offset+a*separation_coef for i in range(n+ind+1)],[amp-i*amp*.1 for i in range(n+ind+1)],marker='$*$',color='black', alpha = .3)


ax[0,0].set_title('MSE-trained',fontsize=20)
ax[0,0].set_ylim([-4e-4,4e-4])
ax[0,1].set_title('EMD-trained',fontsize=20)
ax[0,1].set_ylim([-4e-4,4e-4])
ax[0,2].set_title('FRS-trained',fontsize=20)
ax[0,2].set_ylim([-.1,.1])
ax[1,1].set_ylim([-4e-2,4e-2])
ax[1,0].set_ylim([-4e-2,4e-2])
ax[1,2].set_ylim([-4e-2,4e-2])
ax[2,1].set_ylim([-4e-5,4e-5])
ax[2,0].set_ylim([-4.5e-4,4.5e-4])
ax[2,2].set_ylim([-.4,.4]);

In [ ]:
fig.tight_layout()
fig.savefig('../figures/results_allen_per_animal.pdf', bbox_inches = 'tight')

In [ ]:
all_metrics = ['MSE(id)-MSE(ood)', 'EMD(id)-EMD(ood)', 'EV(id)-EV(ood)']
fig, ax = plt.subplots(len(all_metrics),1,figsize=(5,12))

separation_coef = .5
offset = 1
significance_threshold_1 = 5e-2
significance_threshold_2 = 1e-2
significance_threshold_3 = 1e-3
significance_threshold_4 = 1e-4

for i in range(len(all_metrics)):

    reshaped_mse = results_mse[:,i].reshape(32,100,3).swapaxes(0,2)
    results_mse_per_animal = reshaped_mse[1]-reshaped_mse[2]
    reshaped_emd = results_emd[:,i].reshape(32,100,3).swapaxes(0,2)
    results_emd_per_animal = reshaped_emd[1]-reshaped_emd[2]
    reshaped_frs = results_frs[:,i].reshape(32,100,3).swapaxes(0,2)
    results_frs_per_animal = reshaped_frs[1]-reshaped_frs[2]
    if i == 2:
        #t_stat, p_value_mse = scipy.stats.wilcoxon(results_mse_per_animal.median(dim=1).values,alternative='greater')
        #t_stat, p_value_emd = scipy.stats.wilcoxon(results_emd_per_animal.median(dim=1).values,alternative='greater')
        t_stat_mse, p_value_mse = paired_permutation_test(reshaped_mse[1].median(dim=0)[0],reshaped_mse[2].median(dim=0)[0],alternative='greater')
        t_stat_emd, p_value_emd = paired_permutation_test(reshaped_emd[1].median(dim=0)[0],reshaped_emd[2].median(dim=0)[0],alternative='greater')
        amp = 5e-5
        shift = 2e-6
    else:
        #t_stat, p_value_mse = scipy.stats.wilcoxon(results_mse_per_animal.median(dim=1).values,alternative='less')
        #t_stat, p_value_emd = scipy.stats.wilcoxon(results_emd_per_animal.median(dim=1).values,alternative='less')
        t_stat_mse, p_value_mse = paired_permutation_test(reshaped_mse[1].median(dim=0)[0],reshaped_mse[2].median(dim=0)[0],alternative='less')
        t_stat_emd, p_value_emd = paired_permutation_test(reshaped_emd[1].median(dim=0)[0],reshaped_emd[2].median(dim=0)[0],alternative='less')
        if i == 0:
            amp = 6e-5
            shift = 1e-6
        else:
            amp = 0
            shift = 7e-4

    if p_value_emd < significance_threshold_4:
        n_stars = 4
        ax[i].scatter([offset+separation_coef for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    elif p_value_emd < significance_threshold_3:
        n_stars = 3
        ax[i].scatter([offset+separation_coef for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    elif p_value_emd < significance_threshold_2:
        n_stars = 2
        ax[i].scatter([offset+separation_coef for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    elif p_value_emd < significance_threshold_1:
        n_stars = 1
        ax[i].scatter([offset+separation_coef for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    
    if p_value_mse < significance_threshold_4:
        n_stars = 4
        ax[i].scatter([offset for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    elif p_value_mse < significance_threshold_3:
        n_stars = 3
        ax[i].scatter([offset for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    elif p_value_mse < significance_threshold_2:
        n_stars = 2
        ax[i].scatter([offset for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    elif p_value_mse < significance_threshold_1:
        n_stars = 1
        ax[i].scatter([offset for i in range(n_stars)],[amp-i*shift for i in range(n_stars)],marker='$*$',color='black', alpha = .3)
    
    ax[i] = make_violin(ax[i],results_mse_per_animal.median(dim=1).values,'darkolivegreen',separation_coef=separation_coef)
    ax[i].set_xticks([])
    ax[i].set_ylabel(all_metrics[i],fontsize=20)
    ax[i].ticklabel_format(axis='both', style='sci', scilimits=(-2,1))
    ax[i].hlines(0,.75,1.75,linestyles='dashed',colors='grey')
    ax[i] = make_violin(ax[i],results_emd_per_animal.median(dim=1).values,'blue',separation_coef=separation_coef,offset=1.5)
    #ax[i,2].set_xlim([separation_coef,results_mse_per_animal.shape[1]*separation_coef+1])
    #ax[i] = make_violin(ax[i],results_frs_per_animal.median(dim=1).values,'orange',separation_coef=separation_coef,offset=2)
    ax[i].set_xlim([.7,1.8])
ax[0].set_title('Individual average',fontsize=20)
ax[2].set_xticks([1,1.5])
ax[2].set_xticklabels(['MSE','EMD'],fontsize=16)
fig.tight_layout()

In [ ]:
fig.savefig('../figures/results_allen_per_animal_averaged.pdf', bbox_inches = 'tight')

In [ ]:
all_metrics = ['MSE(id)-MSE(ood)', 'EMD(id)-EMD(ood)', 'EV(id)-EV(ood)']
fig, ax = plt.subplots(len(all_metrics),3,figsize=(10,10))
for i in range(len(all_metrics)):

    ax[i,0] = make_violin(ax[i,0],results_mse[:,i],'darkolivegreen')
    #ax[i,0].set_ylim([min(mean_mse)-2*max(std_mse), max(mean_mse)+2*max(std_mse)])
    ax[i,0].set_xticks([])
    ax[i,0].set_ylabel(all_metrics[i],fontsize=16)
    ax[i,0].ticklabel_format(axis='both', style='sci', scilimits=(-2,1))

    ax[i,1] = make_violin(ax[i,1],results_emd[:,i],'blue')
    ax[i,1].set_xticks([])
    ax[i,1].ticklabel_format(axis='both', style='sci', scilimits=(-2,1))
    
    ax[i,2] = make_violin(ax[i,2],results_frs[:,i],'orange')
    ax[i,2].set_xticks([])
    ax[i,2].ticklabel_format(axis='both', style='sci', scilimits=(-2,1))

ax[0,0].set_title('MSE-trained',fontsize=16)
ax[0,1].set_title('EMD-trained',fontsize=16)
ax[0,2].set_title('FRS',fontsize=16)
ax[i,0].set_xticks([1,1.5,2], ['training set', 'same image', 'different image'], rotation=45)
ax[i,1].set_xticks([1,1.5,2], ['training set', 'same image', 'different image'], rotation=45)
ax[i,2].set_xticks([1,1.5,2], ['training set', 'same image', 'different image'], rotation=45);

In [ ]:
fig.savefig('../figures/results_allen_with_trainset.pdf', bbox_inches = 'tight')

In [ ]:
all_metrics = ['MSE(id)-MSE(ood)', 'EMD(id)-EMD(ood)', 'EV(id)-EV(ood)']
all_results = [results_mse, results_emd, results_frs]
all_colors = ['darkolivegreen', 'blue', 'orange']
fig, ax = plt.subplots(len(all_metrics),3,figsize=(20,12))

separation_coef = .5
offset = 1
significants = [5e-2,1e-2,1e-3,1e-4]

p_values = np.zeros([20,3,3])
for i in range(len(all_metrics)):
    for ind_results, results in enumerate(all_results):
        reshaped = results[:,i].reshape(32,20,5,3).swapaxes(0,3).flatten(start_dim=2).swapaxes(1,2)
        ax[i,ind_results] = make_violin(ax[i,ind_results],reshaped[1]-reshaped[2],all_colors[ind_results], separation_coef=separation_coef, offset=offset)
        for a in range(p_values.shape[0]):
            if i == 2:
                t_stat, p_value = paired_permutation_test(reshaped[1,:,a],reshaped[2,:,a],alternative='greater')
            else:
                t_stat, p_value = paired_permutation_test(reshaped[1,:,a],reshaped[2,:,a],alternative='less')
            p_values[a,i,ind_results] = p_value
        ax[i,ind_results].set_xticks([])
        ax[i,ind_results].ticklabel_format(axis='both', style='sci', scilimits=(-2,1))
        ax[i,ind_results].hlines(0,0,p_values.shape[0]*separation_coef+1,linestyles='dashed',colors='grey')
        ax[i,ind_results].set_xlim([separation_coef,p_values.shape[0]*separation_coef+1])
        ax[2,ind_results].set_xlabel('image id',fontsize=20)
    ax[i,0].set_ylabel(all_metrics[i],fontsize=20)

pvals_fdr_frs = scipy.stats.false_discovery_control(p_values[:,[0,2],2].flatten(), method="bh").reshape(p_values.shape[0],2)
pvals_fdr_seq = scipy.stats.false_discovery_control(p_values[:,:,[0,1]].flatten(), method="bh").reshape(p_values.shape[0],3,2)

amplitudes = np.array([[3.5e-4, 3.5e-4], [3.5e-2,3.5e-2], [4e-4, 3.5e-5]])

for i_loss in range(pvals_fdr_seq.shape[1]):
    for i_train in range(pvals_fdr_seq.shape[2]):
        amp = amplitudes[i_loss,i_train]
        for a in range(p_values.shape[0]):
            n = 4
            ind = -1
            signific = significants[ind]
            while pvals_fdr_seq[a,i_loss,i_train]>significants[ind] and ind+n>0:
                ind-=1
            if pvals_fdr_seq[a,i_loss,i_train]<significants[0]:
                ax[i_loss,i_train].scatter([offset+a*separation_coef for i in range(n+ind+1)],[amp-i*amp*.1 for i in range(n+ind+1)],marker='$*$',color='black', alpha = .3)

for i_loss in range(pvals_fdr_frs.shape[1]):
    amp = 3.5e-1 if i_loss==1 else .085
    for a in range(p_values.shape[0]):
        n = 4
        ind = -1
        while pvals_fdr_frs[a,i_loss]>significants[ind] and ind+n>0:
            ind-=1
        if pvals_fdr_frs[a,i_loss]<significants[0]:
            ax[i_loss*2,2].scatter([offset+a*separation_coef for i in range(n+ind+1)],[amp-i*amp*.1 for i in range(n+ind+1)],marker='$*$',color='black', alpha = .3)

ax[0,0].set_title('MSE-trained',fontsize=20)
ax[0,0].set_ylim([-4e-4,4e-4])
ax[0,1].set_title('EMD-trained',fontsize=20)
ax[0,1].set_ylim([-4e-4,4e-4])
ax[0,2].set_title('FRS-trained',fontsize=20)
ax[0,2].set_ylim([-.1,.1])
ax[1,1].set_ylim([-4e-2,4e-2])
ax[1,0].set_ylim([-4e-2,4e-2])
ax[1,2].set_ylim([-4e-2,4e-2])
ax[2,1].set_ylim([-4e-5,4e-5])
ax[2,0].set_ylim([-4.5e-4,4.5e-4])
ax[2,2].set_ylim([-.4,.4]);

In [ ]:
fig.savefig('../figures/results_allen_per_image.pdf', bbox_inches = 'tight')

In [ ]:
def classifier(dataset_name,training_parameters,metric_names,frs=False,saving_path = 'results/allen_',device='cpu'):

    mse_loss = torch.nn.MSELoss(reduction='none')
    emd_loss = WassDist(zeros='same',normalize=True, reduction = 'none')

    dataset_path = '../../'+dataset_name+'/*'
    print(dataset_path)

    all_testsets, labels = make_allen_testsets_for_decoding(dataset_path)
    print(len(all_testsets))

    all_mse, all_emd = [], []

    for ind_animal in tqdm(range(len(all_testsets))):

        ind_images = labels[ind_animal]

        if frs:
            testing_set = all_testsets[ind_animal].mean(dim=-1).unsqueeze(-1).to(device)
        else:
            testing_set = all_testsets[ind_animal].to(device)

        mse_losses, emd_losses = torch.zeros([20,40]), torch.zeros([20,40])

        num_samples, num_neurons, num_timesteps = testing_set.shape
        training_parameters['kernel_size'] = (1,num_neurons, num_timesteps)
        for image in range(20):
            model = WassA(training_parameters,device=device)
            #print(str(int(image*ind_animal*5)))
            model_name = saving_path+dataset_name+str(int(ind_animal*100+image*5))+get_training_parameters(training_parameters)
            #print(model_name)
            model, training_metrics, _ = learn_motifs(model,testing_set,testing_set,training_parameters,model_name,metric_names,None,verbose=False)
            if training_parameters['normalize_input']:
                testing_set = torch.nn.functional.normalize(testing_set, p=1, dim=2)
            factors_testing, testingset_hat = model(testing_set)
        
            mse_losses[image]  = mse_loss(testingset_hat,testing_set).mean(axis=(1,2)).detach()
            emd_losses[image]  = emd_loss(testingset_hat,testing_set).mean(axis=(1,2)).detach()

        all_mse += [mse_losses]
        all_emd += [emd_losses]
        
    return all_mse, all_emd

In [ ]:
from dataset_generation import kfold_dataset

def make_allen_testsets_for_decoding(dataset_path, number_samples_per_image=10, number_folds=5, device='cpu'):
    
    files_list = glob.glob(dataset_path)
    number_animals = len(files_list)
    all_trainsets, all_testsets = [], []
    id_image_by_animal = []

    for f in range(len(files_list)):
        data = torch.load(files_list[f], weights_only = True)
        data = data.to(torch.float32).to(device)
        number_samples, number_neurons, number_timesteps = data.shape
        number_images = number_samples//number_samples_per_image
        id_image, testsets_animal = [], []
        for image in range(number_images):
            trainsets, testsets, indices = kfold_dataset(data[image*number_samples_per_image:(image+1)*number_samples_per_image],k=number_folds)
            testsets_animal += testsets[0].unsqueeze(0)
            id_image += [image]*testsets[0].shape[0]
        id_image_by_animal += [id_image]
        all_testsets += [torch.cat(testsets_animal)]
            
    return all_testsets, id_image_by_animal

In [ ]:
all_testsets, ids = make_allen_testsets_for_decoding(dataset_path)

In [ ]:
mse, emd = classifier(dataset_name,training_parameters_emd,metric_names,saving_path=saving_path,device=device)

In [ ]:
plt.imshow(mse[2])

In [ ]:
plt.imshow(emd[0])

In [ ]:
for a in range(len(emd)):
    for i in range(emd[0].shape[1]):
        true_ind = i//2
        emd[:,]

In [ ]:
mse, emd = classifier(dataset_name,training_parameters_mse,metric_names,saving_path=saving_path,device=device)

In [ ]:
plt.imshow(mse[2])

In [ ]:
plt.imshow(emd[2])